In [1]:
!pip install pyspark

In [2]:
# Initiate PySpark session
from pyspark.sql import SparkSession

# Create a Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_PySpark_MapReduce") \
    .getOrCreate()

# Check to see if it worked
spark.getActiveSession()

In [3]:
# Download wordcount.txt untuk studi kasus
!wget https://github.com/nivdul/spark-in-practice-scala/blob/master/data/wordcount.txt

--2026-04-09 03:07:07--  https://github.com/nivdul/spark-in-practice-scala/blob/master/data/wordcount.txt
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘wordcount.txt’

wordcount.txt           [ <=>                ] 246.05K  --.-KB/s    in 0.006s  

2026-04-09 03:07:07 (41.8 MB/s) - ‘wordcount.txt’ saved [251959]



In [10]:
# CONTOH #1: MENGHITUNG FREKUENSI KATA DENGAN MAPREDUCE

# Load text file ke format RDD (https://spark.apache.org/docs/latest/rdd-programming-guide.html#resilient-distributed-datasets-rdds)
inputRDD = spark.sparkContext.textFile("wordcount.txt")

# Cek jumlah partisi/split
print('The number of partitions: ',inputRDD.getNumPartitions(), '\nThe total number of elements: ', inputRDD.count())

# Print isi 10 pertama
print(inputRDD.take(10))

The number of partitions:  2 
The total number of elements:  1491
['', '', '', '', '', '', '<!DOCTYPE html>', '<html', '  lang="en"', '  ']


In [16]:
# Flat Mapper mengembalikan flat data dengan separator spasi
words = inputRDD.flatMap(lambda x: x.split(' '))
print(words.take(10))

# MAPPER untuk menyimpan nilai 1 untuk setiap kemunculan
wordsOne = words.map(lambda x: (x, 1))
print(wordsOne.take(10))

['', '', '', '', '', '', '<!DOCTYPE', 'html>', '<html', '']
[('', 1), ('', 1), ('', 1), ('', 1), ('', 1), ('', 1), ('<!DOCTYPE', 1), ('html>', 1), ('<html', 1), ('', 1)]


In [18]:
from operator import add

# REDUCE untuk menjumlahkan (ADD) kemunculan setiap kata yang sudah disimpan di MAPPER
wordCounts = wordsOne.reduceByKey(add)
#print(wordCounts.take(10))

In [20]:
# Pengumpulan hasil (merging hasil-hasil dari reducer)
output = wordCounts.collect()
# Print top-10
output[:10]

[('', 5688),
 ('html>', 1),
 ('<html', 1),
 ('data-color-mode="auto"', 1),
 ('data-dark-theme="dark"', 1),
 ('data-a11y-animated-images="system"', 1),
 ('data-a11y-link-underlines="true"', 1),
 ('>', 23),
 ('<head>', 1),
 ('<link', 33)]

In [21]:
# Tampilkan top 10 terurut
sorted(output, key=lambda x: x[1], reverse = True)[0:10]

[('', 5688),
 ('0', 2991),
 ('1', 776),
 ('data-view-component="true"', 175),
 ('1.75', 155),
 ('aria-hidden="true"', 142),
 ('16', 134),
 ('viewBox="0', 129),
 ('class="octicon', 129),
 ('crossorigin="anonymous"', 120)]

In [22]:
# CONTOH #2: MAPREDUCE dengan data berat ikan
import kagglehub, os

os.environ['KAGGLEHUB_CACHE'] = "/content/kaggle"
# Download fish dataset
path = kagglehub.dataset_download("vipullrathod/fish-market")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fish-market' dataset.
Path to dataset files: /kaggle/input/fish-market


In [24]:
import pyspark.sql.functions as F

# 1. Load the fish.csv ke dataframe
df = spark.read.csv(path + "/Fish.csv", header=True, inferSchema=True)

# 2. Cari ikan terberat dengan groupby (bukan MapReduce)
max_weight_per_species = df.groupBy("Species").agg(F.max("Weight").alias("MaxWeight"))

# 3. Tampilkan hasil
max_weight_per_species.show()

+---------+---------+
|  Species|MaxWeight|
+---------+---------+
|    Roach|    390.0|
|    Smelt|     19.9|
|   Parkki|    300.0|
|Whitefish|   1000.0|
|     Pike|   1650.0|
|    Bream|   1000.0|
|    Perch|   1100.0|
+---------+---------+



In [ ]:
# TUGAS
# Cari 10 species ikan terberat menggunakan map-reduce seperti di contoh #1 wordcount.
# Pastikan daftar species sama dengan yang dihasilkan contoh dengan dataframe (tanpa mapreduce).
# Tips: load dataframe (df) ke spark RDD terlebih dahulu.
# Buat dan upload laporan dalam bentuk 1 halaman PDF yang mencakup strategi kamu di fungsi map dan reduce, kode dan penjelasannya.